# TalentCLEF TaskA 2025

Descripción del corpus: https://talentclef.github.io/talentclef/docs/talentclef-2025/data/description_corpus/

En la evaluación, tenemos tres archivos:

corpus_elements: Conjunto nuevo de trabajos sobre el cual debemos encontrar la similitud frente a uno dado

queries: solicitudes que pretenden encontrar, contrastando con todos los de corpus_elements, indicar si existe similitud (1) o no (0). Para eso habrá que definir un threshold a futuro para ver cuales colapsan a 1 y cuales a 0.

qrels: relaciones entre las queries y cada uno de los elementos del corpus. Solo se muestran los que si presentan relacion (cuarta columna, indicando con un 1).


Los datos de las queries pueden o no pertenecer al corpus conocido (la idea es que no necesariamente, y se construya pensando que no va a haber una solicitud en nuestro corpus). He encontrado algunos casos en los que si aparece en ambos. Ejemplo: lawyer

# Carga de datos

In [1]:
import pandas as pd
import numpy as np

In [46]:
training_data = pd.read_csv("./data/TaskA/training/english/taskA_training_en.tsv", sep='\t', names=['family_id', 'id', 'title1', 'title2'])

validation_data = dict()

validation_data['corpus_elements'] = pd.read_csv("./data/TaskA/validation/english/corpus_elements", sep='\t').set_index('c_id')
validation_data['queries'] = pd.read_csv("./data/TaskA/validation/english/queries", sep='\t').set_index('q_id')
validation_data['qrels'] = pd.read_csv("./data/TaskA/validation/english/qrels.tsv", sep='\t', names=['q_id', 'iter', 'c_id', 'relevance'])

In [17]:
print("Training data samples:")
print(training_data[['title1', 'title2']].head())
print("\n")

print("Validation data samples:")
print(validation_data['corpus_elements'].head())
print(validation_data['queries'].head())
print(validation_data['qrels'].head())

Training data samples:
                        title1                       title2
0                air commodore            flight lieutenant
1  command and control officer               flight officer
2                air commodore  command and control officer
3                pilot officer              squadron leader
4       royal airforce officer  command and control officer


Validation data samples:
   c_id                          jobtitle
0     1                recording engineer
1     2              director of taxation
2     3  technical support representative
3     4                        hr manager
4     5           computer graphic artist
   q_id             jobtitle
0     1                nanny
1     2    food technologist
2     3   broadcast engineer
3     4  automation engineer
4     5         veterinarian
   q_id  iter  c_id  relevance
0     1     0   143          1
1     1     0   150          1
2     1     0   764          1
3     1     0   870          1
4     1  

# Aproximación con embedding ya preentrenado

In [31]:
from sentence_transformers import SentenceTransformer, util

# Version multilingue del modelo
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device='cpu')

## Ejemplo

In [ ]:
# 3. Definir los dos títulos a comparar
title1 = "profesor de colegio"
title2 = "kindergarten teacher"

# 4. Generar los embeddings (representaciones vectoriales)
emb1 = model.encode(title1, convert_to_tensor=True, device='cpu')
emb2 = model.encode(title2, convert_to_tensor=True, device='cpu')

# 5. Calcular la similitud coseno entre ambos
similarity = util.cos_sim(emb1, emb2)

print(f"Similitud entre '{title1}' y '{title2}': {similarity.item():.4f}")

Similitud entre 'profesor de colegio' y 'kindergarten teacher': 0.7835


## Caso de uso

In [39]:
def similarity_between_titles(title1, title2):
    emb1 = model.encode(title1, convert_to_tensor=True, device='cpu')
    emb2 = model.encode(title2, convert_to_tensor=True, device='cpu')

    similarity = util.cos_sim(emb1, emb2)

    return similarity.item()

similarity_between_titles("data scientist", "científico de datos")

0.963962197303772

In [83]:
print('Total de casos a revisar:', validation_data['qrels'].shape[0] * validation_data['queries'].shape[0])

Total de casos a revisar: 254100


In [ ]:
results = []
corpus_size = validation_data['corpus_elements'].shape[0]
    
for q_id in validation_data['queries'].index:
    query = validation_data['queries'].loc[q_id, 'jobtitle']

    for row in validation_data['corpus_elements'].itertuples():
        corpus_element = row.jobtitle
        sim = similarity_between_titles(query, corpus_element)
        results.append({'q_id': q_id, 'c_id': row.Index, 'similarity': sim})
        
        if row.Index % 500 == 0:
            print(f"Processed {row.Index} / {corpus_size} corpus elements for query {q_id}")
            
        break # Remove this line to process all corpus elements
    
print("Done.")

results_df = pd.DataFrame(results)

Done.


In [ ]:
results_df.query('similarity >= 0.55')

,q_id,c_id,similarity
2,3,1,0.729559
9,10,1,0.571719
52,53,1,0.660060
74,75,1,0.615036
87,88,1,0.588437
97,98,1,0.604017


# Modelos de ranking

In [ ]:
from sentence_transformers import CrossEncoder
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
scores = model.predict([("nanny", "recording engineer"), ("nanny", "food technologist")])
